# Replicate ChatGPT Price Forecasting

This notebook gives a concise, reader-friendly tour of the project workflow and the core outputs used in the replication.

## Summary
The Lopez-Lira and Tang (2023) paper "Can ChatGPT Forecast Stock Price Movements? Return Predictability and Large Language Models" documents the capability of LLMs, such as ChatGPT, to predict stock market reactions from news headlines without direct financial training. The papers results suggest forecasting ability generally increases with model size and strategy returns decline as LLM usage within the finanical domain increases. We aim to replicate their results, specifically looking at strategy hit rates, and portfolio returns results.

## Methodology at a glance
- Data foundation: pull CRSP prices and RavenPack headlines from WRDS.
- Data cleaning: map entities to tickers, remove low-quality/duplicate signals, and align headline timing to trading logic.
- NLP labeling: submit batched headlines to OpenAI and parse labels into structured sentiment scores.
- Portfolio construction: convert firm-day sentiment into long/short return series under multiple sample restrictions.
- Reporting: generate summary tables and figures used in the write-up.

## Pipeline context for this notebook
The data shown below comes from the `doit` pipeline in `dodo.py`.
- `pull:crsp_stock` runs `pull_CRSP_stock.py` and writes `CRSP_stock_daily.parquet` and `CRSP_unique_tickers.parquet`.
- `pull:ravenpack` runs `pull_ravenpack.py` and writes `RAVENPACK.parquet`.
- `clean_data:clean_ravenpack` runs `clean_ravenpack.py` and writes `RAVENPACK_cleaned.parquet`.
- `process:generate_batched_requests`, `process:submit_headlines_to_openai`, and `process:create_firm_day_score` produce `daily_headline_polarity.parquet`.
- Downstream scripts `create_portfolios.py` and `create_table1.py` generate return panels and final table CSVs used in this guide.

## Step 0. Setup and Key Paths
Start by confirming which pipeline artifacts exist before loading data. This prevents downstream errors and makes the notebook reproducible across machines and collaborators.

The code imports project settings from `settings.py`, builds the canonical file map, and prints an existence table so we can verify which stages of the pipeline have already been run.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from notebook_helper import (
    generate_single_request_jsonl,
    get_rp_timing_stats,
    printable_table,
)
from settings import config

MANUAL_DATA_DIR = Path(config("MANUAL_DATA_DIR"))
DATA_DIR = Path(config("DATA_DIR"))
OUTPUT_DIR = Path(config("OUTPUT_DIR"))

paths = {
    "CRSP Stock Data": DATA_DIR / "CRSP_stock_daily.parquet",
    "RavenPack Full": DATA_DIR / "RAVENPACK.parquet",
    "RavenPack Clean": DATA_DIR / "RAVENPACK_cleaned.parquet",
    "Daily Headline Scores": DATA_DIR / "daily_headline_polarity.parquet",
    "Portfolio Returns": DATA_DIR / "portfolio_daily_returns.parquet",
    "Table1 (Oct 2021 - May 2024)": DATA_DIR / "table1_overnight_paper_sample.csv",
    "Table1 (Oct 2021 - March 2026)": DATA_DIR / "table1_overnight_full_sample.csv",
    "Paper Table Data": MANUAL_DATA_DIR / "paper_table1.csv",
}

pd.DataFrame(
    {
        "file": list(paths.keys()),
        "exists": [p.exists() for p in paths.values()],
    }
)

## Step 1. Load Cleaned RavenPack and CRSP

In order to replicate the results in this paper, we need market data from CRSP and news data from RavenPack. We are ignoring intraday data. Additionally, we use the RPA Entity Mapping File (wrds_rpa_company_mappings) in order to facilitate the mapping of RavenPack entities to company tickers.

We pulled data between 2021-10-1 and 2026-03-01 for this project. Our replication is concerned with the paper's timebounds of 2021-10-01 to 2024-05-31, and additional analysis was completed on the full sample.

- The <b>CRSP dataset</b> contains stock market data, including prices, returns, and shares outstanding, for securities traded on major U.S. exchanges. This dataset is essential for measuring market performance and calculating variables such as market equity.
- The <b>RavenPack dataset</b>, using over 40,000 sources, provides real-time news analytics, including sentiment analysis and event data focused on business and financial applications. Data includes news and social media content, allowing for comprehensive analysis of financial markets.
- The <b>RPA Entity Mapping File</b> provides a variety of security identifiers (ex. ISINs, CUSIPs, etc.) to allow you to identify the 90,000+ companies.

**Pipeline step that generated this data**
- `CRSP_stock_daily.parquet` is generated by `pull:crsp_stock` via `pull_CRSP_stock.py`.
- `RAVENPACK_cleaned.parquet` is generated by `clean_data:clean_ravenpack` via `clean_ravenpack.py` (after `pull:ravenpack`).

Our pulled CRSP dataset is limited to the world of stocks specified in the paper:

```SQL
WHERE 
    ( 
        primaryexch IN ( 'N', 'A', 'Q' ) AND
        conditionaltype = 'RW' AND
        tradingstatusflg = 'A' AND
        dlycaldt >= '{start_date}' AND
        dlycaldt <= '{end_date}'
        {permno_filter}
    ) ;
```

The `clean_ravenpack` process was designed to replicate the data cleaning procedure described in the paper. This includes filtering RavenPack data to match CRSP tickers, ensure a baseline relevancy rating of 0.60, deduplicating firm-day headlines using Optimal String Alignment (OSA) similarity, and aligning headline timestamps to Eastern Time. Intraday headlines are excluded, and overnight headlines are adjusted to ensure proper alignment with trading day per teh overnight cutoff at 4:00PM ET. This step ensures that the cleaned dataset is consistent with the methodology outlined in the paper.

In [ ]:
rp = (
    pd.read_parquet(paths["RavenPack Clean"])
    if paths["RavenPack Clean"].exists()
    else pd.DataFrame()
)
crsp = (
    pd.read_parquet(paths["CRSP Stock Data"])
    if paths["CRSP Stock Data"].exists()
    else pd.DataFrame()
)

print("RavenPack cleaned shape:", rp.shape)
print("CRSP Stock Data shape:", crsp.shape)

display(rp.head(3))
display(crsp.head(3))
del crsp

### Headline Timing Diagnostics

In [ ]:
stats, table = get_rp_timing_stats(rp)

display(stats.style)
display(table.style)

## Step 3. Generate Batched Requests (Single-Row Example)

**Pipeline step that generated this data**
- `openai_headline_requests.*.jsonl` files are generated by `process:generate_batched_requests` via `generate_batched_requests.py`.
- `id_to_row_mapping.*.json` files are generated by `process:generate_batched_requests` via `generate_batched_requests.py`.

This step demonstrates the `generate_batched_requests.py` logic on a single RavenPack row to replicate the pipeline behavior.

In order to retrieve OpenAI responses to our headline prompts, we must create batch files for asynchornous submission. The RavenPack data is broken up into chunks (40,000 being the default) and each request is a single line of json (referred to as JSONL). We also create a separate mapping file for ease of processing later on.

In [ ]:
jsonl_content, mapping_content = generate_single_request_jsonl(rp.head(1).copy())

In [ ]:
print("JSONL content inline:")
display(jsonl_content)

In [ ]:
print("JSONL content formatted:")
print(
    json.dumps(
        json.loads(jsonl_content), sort_keys=True, indent=2, separators=(",", ": ")
    )
)

In [ ]:
print("ID to Row Mapping:")
print(
    json.dumps(
        json.loads(mapping_content), sort_keys=True, indent=2, separators=(",", ": ")
    )
)

## Step 4. Submit Headlines to OpenAI Batch API

**Pipeline step that generated this data**
- `openai_headline_batch_output.*.jsonl` files are generated by `process:submit_headlines_to_openai` via `submit_headlines_to_openai.py`.
- `openai_headline_batch_metadata.*.json` files are generated by `process:submit_headlines_to_openai` via `submit_headlines_to_openai.py`.

All batch request files matching `openai_headline_requests.*.jsonl` are then submitted to OpenAI via the `process:submit_headlines_to_openai` step, with an SLA of 24 hours for a response. The pipeline blocks while waiting for a completion status for all submitted files. Results are written to the `DATA_DIR` as `openai_headline_batch_output.*.jsonl` with corresponding metadata files `openai_headline_batch_metadata.*.json`.

We prompt the LLM for one of three responses
- `YES`: headline was deemed positive for stock price forecast
- `NO`: headline was deemed negative for stock price forecast
- `UNKNOWN`: headline was deemed neutral for stock price forecast

Below is a success response from OpenAI.

In [ ]:
data = pd.read_json(DATA_DIR / "openai_headline_batch_output.1.jsonl", lines=True).head(
    1
)
display(data.iloc[0]["response"])

## Step 5. Firm-Day Sentiment Output

The next step in our pipeline processes the OpenAI responses, and aggregates the model's rating of each headline.

**Pipeline step that generated this data**
- `daily_headline_polarity.parquet` is generated by `process:create_firm_day_score` in `create_firmday_score.py`

We parse each response, and if the answer for a headline was `YES` it gets a score of 1, `NO` gets a score of -1, and `UNKNOWN` gets a score of 0. We group the data set by date and ticker, and aggregate the total headlines, total score sum, and overall polarity (positive or negative). The results are written to `DATA_DIR` as `daily_headline_polarity.parquet`

In [ ]:
import create_firmday_score as cfs

cfs.main()

In [ ]:
scores = (
    pd.read_parquet(paths["Daily Headline Scores"])
    if paths["Daily Headline Scores"].exists()
    else pd.DataFrame()
)
print("Scores shape:", scores.shape)
display(scores.head(5).style)

if not scores.empty:
    summary = (
        scores["score"]
        .value_counts(dropna=False)
        .rename_axis("score")
        .reset_index(name="count")
    )
    summary["share"] = (summary["count"] / summary["count"].sum()).round(4)
    summary.sort_values("score", inplace=True, ignore_index=True)
    summary = summary.rename(index={0: "NO", 1: "UNKNOWN", 2: "YES"})
    display(summary.style)

## Step 6. Portfolio Returns

**Pipeline step that generated this data**
- `portfolio_daily_returns.parquet` is generated by `process:create_portfolios` via `create_portfolios.py`.

In order to actually replicate the paper results, we need to translate the headline scores into trading signals and build the following daily-rebalanced portfolios as described in the paper:

- **Long-Short Portfolio**: buying positive predictions and selling negative predictions when both legs have at least two firms; otherwise entering just one leg.
- **Long-Only Portfolio**: buying only positive predictions.
- **Short-Only Portfolio**: selling only negative predictions.

Moreover, to replicate Figure 5 from the paper, which shows the cumulative return of the Long-Short portfolio with additional restrictions as described in the paper (we are excluding transaction costs):

- **Not Small**: market capitalization above the 20th percentile
- **Price > 5**: close price greater than $5 on previous day

In [ ]:
port = (
    pd.read_parquet(paths["Portfolio Returns"])
    if paths["Portfolio Returns"].exists()
    else pd.DataFrame()
)
print("Portfolio returns:", port.shape)
display(port.head(5).style)

## Step 7: Generate Table 1

**Pipeline step that generated this data**
- `table1_overnight_full_sample.csv` is generated by `process:create_table1` via `create_table1.py`
- `table1_overnight_paper_sample.csv` is generated by `process:create_table1` via `create_table1.py`

Once we have our portfolios constructed we can attempt to replicate Table 1 from the paper. Additionally, we will generate a version of the table that includes data through January 2026. The source of the portfolio data is `portfolio_daily_returns.parquet` as described above.

In [ ]:
import create_table1 as ct1

ct1.main()

In [ ]:
full_sample_table = (
    pd.read_csv(paths["Table1 (Oct 2021 - March 2026)"])
    if paths["Table1 (Oct 2021 - March 2026)"].exists()
    else pd.DataFrame()
)
paper_sample_table = (
    pd.read_csv(paths["Table1 (Oct 2021 - May 2024)"])
    if paths["Table1 (Oct 2021 - May 2024)"].exists()
    else pd.DataFrame()
)
display(printable_table(full_sample_table.head(5), "Full Sample").style)
display(printable_table(paper_sample_table.head(5), "Paper Sample Replication").style)

### Replication Comparison

In [ ]:
actual_paper_table = (
    pd.read_csv(paths["Paper Table Data"])
    if paths["Paper Table Data"].exists()
    else pd.DataFrame()
)
display(printable_table(actual_paper_table, "Actual Paper Table").style)

In [ ]:
actual = actual_paper_table

compare_cols = [col for col in paper_sample_table.columns if col in actual.columns]

actual_cmp = actual[compare_cols].set_index("Portfolio")
paper_cmp = paper_sample_table[compare_cols].set_index("Portfolio")

numeric_cols = [col for col in compare_cols if col != "Portfolio"]

diff_table = paper_cmp[numeric_cols].apply(pd.to_numeric, errors="coerce") - actual_cmp[
    numeric_cols
].apply(pd.to_numeric, errors="coerce")

display(diff_table.reset_index().style.format({col: "{:.4f}" for col in numeric_cols}))

pct_error_table = pd.DataFrame(index=actual_cmp.index)

for col in numeric_cols:
    actual_vals = pd.to_numeric(actual_cmp[col], errors="coerce")
    paper_vals = pd.to_numeric(paper_cmp[col], errors="coerce")
    pct_error = ((paper_vals - actual_vals) / actual_vals.replace(0, pd.NA)) * 100

    pct_error_table[f"{col} | actual"] = actual_vals
    pct_error_table[f"{col} | paper_sample"] = paper_vals
    pct_error_table[f"{col} | error (%)"] = pct_error

pct_formats = {}
for col in numeric_cols:
    pct_formats[f"{col} | actual"] = "{:.4f}"
    pct_formats[f"{col} | paper_sample"] = "{:.4f}"
    pct_formats[f"{col} | error (%)"] = "{:.2f}%"

error_cols = [c for c in pct_error_table.columns if c.endswith("| error (%)")]
display(
    pct_error_table[error_cols]
    .reset_index()
    .style.format({col: "{:.2f}%" for col in error_cols})
)

### Comparison Discussion

As we can see from the above, some of our replication was fairly consistent with the papers results, specifically Initial Reaction Hit Rate of the Long-Short and Long-Only portfolios as well as the Drift Hit Rate of the Short-Only portfolio. Unfortuantely, everything else was substantially different.

This can almost certainly be attributed to one key fact: the GPT model specified in the paper, `gpt-4-0314`, was deprecated by OpenAI and no logner available. Therefore we opted to use the next most recent model `gpt-3.5-turbo`. Given the paper's results suggesting the older the model the  worse it would perform, it is no surprise that the mdoel we used largely performed worse.

Specifically, we note that our responses from OpenAI using the older model produced significantly more `UNKNOWN` responses than the papers model, i.e. ~67% vs 34.7%

In [ ]:
import create_openai_responses_table as cort

cort.main()